In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 读取 2023_24_Shenzhen.xlsx 文件
shenzhen_file = '2023_24_Shenzhen.xlsx'
if not os.path.exists(shenzhen_file):
    print(f"{shenzhen_file} 文件不存在。")
    exit()

shenzhen_df = pd.read_excel(shenzhen_file)

# 确保Date列为整数格式
shenzhen_df['Date'] = shenzhen_df['Date'].astype(int)

# 提取 2023_24_Shenzhen.xlsx 的数据到一个字典
shenzhen_data = {}
for index, row in shenzhen_df.iterrows():
    shenzhen_data[row['Date']] = {
        'MINT': row['MINT'],
        'MAXT': row['MAXT'],
        'MEAN': row['MEAN']
    }

# 获取当前目录下所有符合 data* 格式的 Excel 文件
excel_files = [f for f in os.listdir() if f.startswith('data') and f.endswith('.xlsx')]

# 添加进度条
for excel_file in tqdm(excel_files, desc="Processing Excel files"):
    try:
        # 读取 Excel 文件
        df = pd.read_excel(excel_file)

        # 确保倒数第一列为 'Date'，倒数第二列为 '时间'
        if df.shape[1] < 2:
            print(f"{excel_file} 没有足够的列，跳过该文件。")
            continue

        # 确保倒数第一列Date为整数格式
        df.iloc[:, -1] = df.iloc[:, -1].astype(int)

        date_col = df.iloc[:, -1]
        time_col = df.iloc[:, -2]

        # 初始化温度列
        temp_col = []

        # 遍历每行数据，根据时间和日期匹配温度
        for date, hour in zip(date_col, time_col):
            if date in shenzhen_data:
                temp_row = shenzhen_data[date]
                if hour in range(0, 6) or hour in range(22, 24):
                    temp_col.append(temp_row['MINT'])  # 清晨和夜间温度
                elif hour in range(6, 12) or hour in range(17, 22):
                    temp_col.append(temp_row['MEAN'])  # 白天和傍晚温度
                elif hour in range(12, 17):
                    temp_col.append(temp_row['MAXT'])  # 中午温度
                else:
                    temp_col.append(None)  # 无法匹配的时间段
            else:
                temp_col.append(None)  # 如果日期不在深圳数据中，温度设置为None

        # 添加温度列到原始数据
        df['Temp'] = temp_col

        # 保存修改后的数据到原文件
        df.to_excel(excel_file, index=False)

        print(f"{excel_file} 已处理完毕。")
    except Exception as e:
        print(f"处理文件 {excel_file} 时出错: {e}")

print("所有文件处理完毕。")
